# Instacart MLlib – Local Run (Jupyter + HDFS)

Notebook này thay thế `05_colab_full_mllib_run.ipynb` để chạy trực tiếp trên **Jupyter localhost:8888** kết nối với cụm **HDFS**.

Yêu cầu môi trường:
- Python ≥ 3.9, Java ≥ 11
- PySpark đã cài (`pip install pyspark==4.1.1`)
- HDFS đang chạy và `hdfs` CLI có trong `PATH`
- Data 6 file CSV Instacart đã upload lên HDFS tại `HDFS_DATA_DIR`

## 0. Cấu hình

In [ ]:
import os
from pathlib import Path

# ============================================================
# TODO: Chỉnh các giá trị dưới đây cho đúng môi trường nhóm
# ============================================================

# Đường dẫn tuyệt đối tới thư mục project trên máy local
# Ví dụ: /home/nhom05/work  hoặc  /home/user/InstaCart-Online-Basket-Analysis
PROJECT_DIR = Path("/home/nhom05/work/InstaCart-Online-Basket-Analysis")

# HDFS namenode URI
HDFS_NAMENODE = "hdfs://namenode:9000"

# Thư mục chứa 6 file CSV trên HDFS
HDFS_DATA_DIR = f"{HDFS_NAMENODE}/user/nhom05/data"

# Thư mục lưu kết quả LOCAL (models, reports, features)
SEED            = 42
SAMPLE_FRACTION = 1.0   # 1.0 = full; 0.02 = smoke-test nhanh
RUN_TAG = "local_hdfs_full_seed42" if SAMPLE_FRACTION == 1.0 else f"local_hdfs_sample_{SAMPLE_FRACTION}_seed42"
OUTPUT_DIR = PROJECT_DIR / "local_outputs" / RUN_TAG

# Spark tuning
TASKS               = "all"      # all | reorder | segmentation | basket
MODELS              = "lr,rf,gbt"
K_MIN               = 2
K_MAX               = 8
MIN_SUPPORT         = 0.003
MIN_CONFIDENCE      = 0.2
DRIVER_MEMORY       = "5g"
SHUFFLE_PARTITIONS  = 32
DEFAULT_PARALLELISM = 16
SPARK_LOCAL_DIR     = "/tmp/spark-tmp"

# Script ML (đã patch để hỗ trợ HDFS)
ML_SCRIPT = PROJECT_DIR / "src" / "03_ml" / "local_train_mllib.py"

# ============================================================
print("PROJECT_DIR  :", PROJECT_DIR)
print("HDFS_DATA_DIR:", HDFS_DATA_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("ML_SCRIPT    :", ML_SCRIPT)
print("RUN_TAG      :", RUN_TAG)

## 1. Kiểm tra môi trường

In [ ]:
import sys, subprocess

print("Python:", sys.version)

# Java
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java  :", (result.stdout or result.stderr).splitlines()[0])

# PySpark
try:
    import pyspark
    print("PySpark:", pyspark.__version__)
except ImportError:
    print("PySpark chưa cài. Chạy: pip install pyspark==4.1.1")

# Kiểm tra SPARK_HOME
spark_home = os.environ.get("SPARK_HOME", "")
if spark_home and not Path(spark_home, "bin", "spark-submit").exists():
    print(f"[WARN] SPARK_HOME không hợp lệ ({spark_home}), xóa để PySpark tự cấu hình.")
    del os.environ["SPARK_HOME"]
else:
    print("SPARK_HOME:", spark_home or "(không đặt – PySpark tự cấu hình)")

# HDFS CLI
r = subprocess.run(["hdfs", "version"], capture_output=True, text=True)
if r.returncode == 0:
    print("HDFS CLI:", r.stdout.splitlines()[0])
else:
    print("[WARN] Không tìm thấy lệnh 'hdfs' trong PATH – kiểm tra HADOOP_HOME")

## 2. Xác nhận project directory

In [ ]:
if not ML_SCRIPT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy script: {ML_SCRIPT}\n"
        "Hãy chỉnh PROJECT_DIR ở cell 0 cho đúng."
    )

print("✓ Project dir tồn tại :", PROJECT_DIR)
print("✓ ML script tồn tại  :", ML_SCRIPT)

## 3. Kiểm tra kết nối HDFS và dữ liệu

In [ ]:
REQUIRED_FILES = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]

# Liệt kê thư mục data trên HDFS
r = subprocess.run(
    ["hdfs", "dfs", "-ls", HDFS_DATA_DIR],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(
        f"Không kết nối được HDFS hoặc thư mục không tồn tại: {HDFS_DATA_DIR}\n"
        f"{r.stderr}"
    )

print("HDFS listing của", HDFS_DATA_DIR)
print(r.stdout)

# Kiểm tra từng file
missing = []
for fname in REQUIRED_FILES:
    hdfs_path = f"{HDFS_DATA_DIR}/{fname}"
    check = subprocess.run(
        ["hdfs", "dfs", "-test", "-e", hdfs_path],
        capture_output=True
    )
    status = "✓" if check.returncode == 0 else "✗ MISSING"
    print(f"  {status}  {fname}")
    if check.returncode != 0:
        missing.append(fname)

if missing:
    raise FileNotFoundError(
        f"Thiếu các file trên HDFS: {missing}\n"
        f"Upload bằng: hdfs dfs -put <file.csv> {HDFS_DATA_DIR}/"
    )

print("\n✓ Tất cả 6 file CSV đã có trên HDFS.")

## 4. Chuẩn bị feature config

In [ ]:
import json

# Tìm feature config – ưu tiên file đã có trong repo
FEATURE_CONFIG = None
candidates = [
    PROJECT_DIR / "notebooks" / "my_work" / "outputs" / "sklearn_research" / "selected_features_for_mllib.json",
    PROJECT_DIR.parent / "notebooks" / "my_work" / "outputs" / "sklearn_research" / "selected_features_for_mllib.json",
    PROJECT_DIR / "src" / "03_ml" / "selected_features_for_mllib.generated.json",
]
for c in candidates:
    if c.exists():
        FEATURE_CONFIG = c
        break

# Nếu không có file nào → tạo config mặc định
if FEATURE_CONFIG is None:
    generated = PROJECT_DIR / "src" / "03_ml" / "selected_features_for_mllib.generated.json"
    generated.parent.mkdir(parents=True, exist_ok=True)
    generated.write_text(json.dumps({
        "source": "generated by 05_local_hdfs_mllib_run.ipynb",
        "best_model": {
            "trial": "hgb_lr0.05_iter160_leaf31",
            "model": "hist_gbdt",
            "average_precision": 0.37601770317532923,
            "roc_auc": 0.8206271797102557,
        },
        "reorder_numeric_features": [
            "up_orders_since_last", "up_order_count", "up_reorder_rate",
            "up_order_rate_since_first", "u_reorder_rate", "p_reorder_rate",
            "u_avg_basket_size", "u_std_days_since_prior", "p_aisle_id",
            "u_dairy_ratio", "p_total_orders", "u_unique_departments",
            "u_total_items", "u_avg_days_since_prior", "u_preferred_hour",
            "up_avg_position", "up_first_order_number", "u_unique_aisles",
        ],
        "reorder_categorical_features": ["p_department_id"],
        "segmentation_features": [
            "recency", "frequency", "volume", "u_reorder_rate",
            "u_organic_ratio", "u_distinct_products", "u_unique_departments",
            "u_produce_ratio", "u_dairy_ratio",
        ],
        "basket": {
            "algorithm": "FPGrowth",
            "sort_rules_by": ["lift", "confidence", "support"],
        },
    }, indent=2), encoding="utf-8")
    FEATURE_CONFIG = generated
    print("[INFO] Tạo feature config mặc định tại:", FEATURE_CONFIG)
else:
    print("✓ Dùng feature config:", FEATURE_CONFIG)

## 5. Chạy MLlib pipeline

In [ ]:
import time

Path(SPARK_LOCAL_DIR).mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(ML_SCRIPT),
    "--data-dir",           HDFS_DATA_DIR,          # HDFS URI
    "--hdfs-namenode",      HDFS_NAMENODE,          # cấu hình fs.defaultFS
    "--output-dir",         str(OUTPUT_DIR),
    "--master",             "local[*]",
    "--tasks",              TASKS,
    "--models",             MODELS,
    "--sample-fraction",    str(SAMPLE_FRACTION),
    "--seed",               str(SEED),
    "--k-min",              str(K_MIN),
    "--k-max",              str(K_MAX),
    "--min-support",        str(MIN_SUPPORT),
    "--min-confidence",     str(MIN_CONFIDENCE),
    "--driver-memory",      DRIVER_MEMORY,
    "--shuffle-partitions", str(SHUFFLE_PARTITIONS),
    "--default-parallelism",str(DEFAULT_PARALLELISM),
    "--local-dir",          SPARK_LOCAL_DIR,
    "--feature-config",     str(FEATURE_CONFIG),
    "--overwrite",
]

print("Command:")
print(" ".join(cmd))
print("\n" + "=" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True)
elapsed = time.time() - start

print("=" * 60)
print(f"Exit code : {result.returncode}")
print(f"Thời gian : {elapsed / 60:.2f} phút")

if result.returncode != 0:
    raise RuntimeError("Pipeline thất bại. Xem log phía trên.")

## 6. Xem kết quả

In [ ]:
summary_path = OUTPUT_DIR / "reports" / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(summary_path)

summary = json.loads(summary_path.read_text(encoding="utf-8"))

if "reorder" in summary:
    best = summary["reorder"]["best_model"]
    print("── Reorder best model ──────────────────")
    for k in ["model", "auc_pr", "auc_roc", "accuracy",
               "precision_pos", "recall_pos", "f1_pos", "tp", "fp", "fn", "tn"]:
        print(f"  {k:20s}: {best.get(k)}")

if "segmentation" in summary:
    seg = summary["segmentation"]
    print("\n── Segmentation ────────────────────────")
    print(f"  user_count     : {seg.get('user_count')}")
    print(f"  best_k         : {seg.get('best_k')}")
    print(f"  best_silhouette: {seg.get('best_silhouette')}")

if "basket" in summary:
    basket = summary["basket"]
    print("\n── Basket / FP-Growth ──────────────────")
    for k in ["basket_count", "freq_itemset_count", "rule_count",
               "min_support", "min_confidence"]:
        print(f"  {k:22s}: {basket.get(k)}")
    print("\n  Top-5 rules:")
    for rule in basket.get("top_rules", [])[:5]:
        ant  = rule["antecedent_names"]
        con  = rule["consequent_names"]
        conf = round(rule["confidence"], 4)
        lift = round(rule["lift"],       4)
        print(f"    {ant} => {con}  conf={conf}  lift={lift}")

## 7. Lưu kết quả (zip local)

In [ ]:
import shutil

zip_base = PROJECT_DIR / "local_outputs" / RUN_TAG
zip_path = shutil.make_archive(
    str(zip_base),   # tên file zip (không có đuôi)
    "zip",
    root_dir=str(OUTPUT_DIR)
)
zip_size_mb = Path(zip_path).stat().st_size / (1024 ** 2)
print(f"ZIP: {zip_path}  ({zip_size_mb:.2f} MB)")
print("Kết quả được lưu tại:", zip_path)

## 8. (Tuỳ chọn) Upload kết quả lên HDFS

In [ ]:
HDFS_OUTPUT_DIR = f"{HDFS_NAMENODE}/user/nhom05/mllib_outputs/{RUN_TAG}"

subprocess.run(["hdfs", "dfs", "-mkdir", "-p", HDFS_OUTPUT_DIR], check=True)
subprocess.run(
    ["hdfs", "dfs", "-put", "-f", zip_path, HDFS_OUTPUT_DIR + "/"],
    check=True
)
print(f"✓ Đã upload {zip_path} lên {HDFS_OUTPUT_DIR}/")

## 9. Chạy lại trên Spark cluster chính thức

Khi cluster master của nhóm mở:

```bash
spark-submit /home/nhom05/work/03_ml/01_reorder_classifier.py
spark-submit /home/nhom05/work/03_ml/02_customer_segmentation.py
spark-submit /home/nhom05/work/03_ml/03_market_basket_fpgrowth.py
```

Hoặc truyền HDFS namenode trực tiếp cho script:

```bash
python local_train_mllib.py \
    --data-dir hdfs://namenode:9000/user/nhom05/data \
    --hdfs-namenode hdfs://namenode:9000 \
    --tasks all --models lr,rf,gbt --seed 42 --overwrite
```